# Vending-Bench 2 — Lambda GPU Setup Verification

This notebook verifies that a **Lambda GH200 instance** is correctly configured to run
the VB2 GRPO training notebook (`01_vb2_training_grpo.ipynb`).

It checks:
1. Dependencies install correctly
2. CUDA/GPU is available (GH200, 480GB VRAM)
3. TRL imports without vllm ABI errors
4. Model loads with 4-bit quantization + LoRA
5. Gradient flow works (forward + backward pass)
6. VB2 environment works — all tool calls verified via direct Python API

## 1 — Install Dependencies

In [3]:
%%capture
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"

!pip install -qqq uv

!uv pip uninstall --system torch torchvision torchaudio torchao torch_c_dlpack_ext unsloth unsloth_zoo vllm 2>/dev/null

!uv pip install --system "numpy<2" "transformers>=4.49,<4.52" "trl>=0.15,<0.17" \
    peft datasets accelerate bitsandbytes \
    openenv-core fastmcp matplotlib

# Remove vllm again in case trl re-pulled it
!uv pip uninstall --system vllm 2>/dev/null || true

## 2 — Verify CUDA / GPU

In [4]:
import torch

assert torch.cuda.is_available(), "CUDA not available!"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)

print(f"torch:    {torch.__version__}")
print(f"CUDA:     {torch.version.cuda}")
print(f"GPU:      {gpu_name}")
print(f"VRAM:     {vram_gb:.0f} GB")

assert "GH200" in gpu_name or vram_gb > 400, f"Expected GH200 with 480GB VRAM, got {gpu_name} with {vram_gb:.0f}GB"
print("\nGPU verification passed.")

AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 3 — Block vllm & Verify TRL Imports

In [ ]:
import sys

# Remove any cached vllm modules
for key in list(sys.modules.keys()):
    if key == "vllm" or key.startswith("vllm."):
        del sys.modules[key]

# Patch TRL's availability check
import trl.import_utils
trl.import_utils._vllm_available = False

from trl import GRPOConfig, GRPOTrainer
print(f"trl:      {trl.__version__}")
print(f"GRPOConfig imported: {GRPOConfig is not None}")
print(f"GRPOTrainer imported: {GRPOTrainer is not None}")
print("\nTRL import verification passed.")

## 4 — Load Model with 4-bit Quantization + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.print_trainable_parameters()
print("\nModel loading verification passed.")

## 5 — Verify Gradient Flow

In [ ]:
# Run a small forward + backward pass to catch gradient issues early
test_input = tokenizer("Hello, world!", return_tensors="pt").to("cuda")
test_input["labels"] = test_input["input_ids"].clone()

output = model(**test_input)
loss = output.loss
loss.backward()

# Check that at least some LoRA parameters got gradients
grad_params = sum(1 for p in model.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
print(f"Loss:                {loss.item():.4f}")
print(f"Params with grads:   {grad_params}")
assert grad_params > 0, "No parameters received gradients!"

model.zero_grad()
print("\nGradient flow verification passed.")

## 6 — VB2 Environment: Import & Reset

In [ ]:
%%capture
!git clone https://github.com/retroam/vendsim-vb2.git /tmp/vendsim-vb2 2>/dev/null || true
import sys
sys.path.insert(0, '/tmp/vendsim-vb2')

In [ ]:
from vendsim_vb2.environment import VendingBench2Environment
from vendsim_vb2.demand import PRODUCTS
import json

env = VendingBench2Environment(seed=42)
env.reset()

print(f"Starting balance: ${env.state.cash_balance}")
print(f"Products: {list(PRODUCTS.keys())}")
print(f"Day: {env.state.day_index}")
assert env.state.cash_balance == 500.0
assert len(PRODUCTS) == 5
print("\nEnvironment reset verification passed.")

## 7 — VB2 Tool Calls: Explore Balance & Inventory

In [ ]:
# Check balance
balance = env.state.cash_balance
print(f"Balance: ${balance}")
assert isinstance(balance, (int, float))

# Check storage inventory
storage_inv = dict(env.state.storage_inventory)
print(f"Storage inventory: {storage_inv}")
assert isinstance(storage_inv, dict)

# Check machine inventory
machine_inv = dict(env.state.machine_inventory)
print(f"Machine inventory: {machine_inv}")
assert isinstance(machine_inv, dict)

print("\nBalance & inventory verification passed.")

## 8 — VB2 Tool Calls: Set Prices

In [ ]:
# Set prices for all products
test_prices = {"soda": 1.75, "water": 1.25, "candy": 1.00, "chips": 2.50, "sandwich": 4.50}

for product, price in test_prices.items():
    env.set_price(product, price)
    current = env.state.prices.get(product)
    print(f"  {product}: set to ${price} -> current: ${current}")
    assert current == price, f"Price mismatch for {product}: expected {price}, got {current}"

print("\nPrice setting verification passed.")

## 9 — VB2 Tool Calls: Supplier Quote & Inventory Restock

In [ ]:
# Request supplier quote
quote = env.request_supplier_quote("chips", 20)
print(f"Supplier quote: {quote}")
assert quote is not None

# Stock storage and restock machine
for product in PRODUCTS:
    env.state.storage_inventory[product] = 20

restock_result = env.run_sub_agent("restock_machine", product="soda", qty=3)
print(f"Restock result: {restock_result}")
assert restock_result is not None

print("\nSupplier & restock verification passed.")

## 10 — VB2 Tool Calls: Scratchpad

In [ ]:
# Write to scratchpad
env.write_scratchpad("Setup verification: all systems nominal.")
print("Scratchpad written.")

# Read scratchpad
notes = env.read_scratchpad()
print(f"Scratchpad contents: {notes}")
assert "Setup verification" in str(notes)

print("\nScratchpad verification passed.")

## 11 — VB2 Tool Calls: Advance Days & Observe Sales

In [ ]:
NUM_DAYS = 5

print(f"Advancing {NUM_DAYS} days...\n")
for i in range(NUM_DAYS):
    result = env.wait_for_next_day()
    sales = result.payload.get('sales', {})
    revenue = result.payload.get('revenue', 0.0)
    print(f"  Day {env.state.day_index - 1}: sales={sales}, revenue=${revenue:.2f}, balance=${env.state.cash_balance:.2f}")

final_balance = env.state.cash_balance
print(f"\nFinal balance after {NUM_DAYS} days: ${final_balance:.2f}")
assert isinstance(final_balance, (int, float))

print("\nDay advancement verification passed.")

## 12 — Summary

In [ ]:
print("="*50)
print(" Lambda GH200 Setup Verification")
print("="*50)
print(f"  [PASS] Dependencies installed")
print(f"  [PASS] CUDA/GPU: {gpu_name}, {vram_gb:.0f}GB VRAM")
print(f"  [PASS] TRL imports (vllm blocked)")
print(f"  [PASS] Model: 4-bit quant + LoRA")
print(f"  [PASS] Gradient flow")
print(f"  [PASS] VB2 environment reset")
print(f"  [PASS] Balance & inventory checks")
print(f"  [PASS] Price setting")
print(f"  [PASS] Supplier quote & restock")
print(f"  [PASS] Scratchpad read/write")
print(f"  [PASS] Day advancement & sales")
print("="*50)
print("\nAll checks passed. Ready to run 01_vb2_training_grpo.ipynb")